[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/01_eda_analysis.ipynb)

In [ ]:
import os, sys
from pathlib import Path

# ── Detectar si estamos en Google Colab ──────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Estrategia 1: acceso por ID de carpeta raíz del proyecto
    _p1 = "/content/drive/.shortcut-targets-by-id/13VEeA8Jt0G_NNpJelWZrhacBhA_mRpnU"
    # Estrategia 2: acceso por ID de TrainData (subir un nivel)
    _p2 = "/content/drive/.shortcut-targets-by-id/13Dj5DYGOwgMGomMC8gCySKGr0ne8hOpU"

    if os.path.exists(_p1) and os.path.isdir(_p1):
        DATA_ROOT_OVERRIDE = _p1
        print(f"Estrategia 1 OK: {_p1}")
    elif os.path.exists(_p2) and os.path.isdir(_p2):
        DATA_ROOT_OVERRIDE = str(Path(_p2).parent)
        print(f"Estrategia 2 OK: {DATA_ROOT_OVERRIDE}")
    else:
        import subprocess
        _r = subprocess.run(
            ["find", "/content/drive/MyDrive", "-name", "TrainData",
             "-type", "d", "-maxdepth", "6"],
            capture_output=True, text=True, timeout=30
        )
        _hits = [p.replace('/TrainData', '') for p in _r.stdout.strip().splitlines() if p]
        if _hits:
            DATA_ROOT_OVERRIDE = _hits[0]
            print(f"Estrategia 3 OK: {DATA_ROOT_OVERRIDE}")
        else:
            print("No se encontró el dataset. Ajusta DATA_ROOT_OVERRIDE manualmente:")
            print("  DATA_ROOT_OVERRIDE = 'ruta/en/tu/Drive'")

    if DATA_ROOT_OVERRIDE:
        print(f"Dataset: {DATA_ROOT_OVERRIDE}")
else:
    print('Entorno local — se usará detección automática de rutas.')


# 01 — Análisis Exploratorio de Datos (EDA)

Análisis exploratorio completo sobre datos **reales** del dataset Landslide4Sense (ISPRS 2022).

Hallazgos clave documentados:
1. Balance de clases casi equiparado (58.7% / 41.3%)
2. Deslizamientos de pequeña escala (área mediana 2.04%)
3. Canales más discriminativos: RedEdge3, RedEdge2, DEM

---

In [ ]:
import os, sys
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL DATASET
# Si la celda de Drive de arriba encontró el dataset, DATA_ROOT_OVERRIDE
# ya está definida. Si no, puedes ajustarla manualmente aquí:
# DATA_ROOT_OVERRIDE = "/content/drive/MyDrive/landslide4sense"
# ══════════════════════════════════════════════════════════════════════════════

# Preservar DATA_ROOT_OVERRIDE si ya fue fijada por la celda de Drive
try:
    DATA_ROOT_OVERRIDE
except NameError:
    DATA_ROOT_OVERRIDE = None

# ── Detección automática ──────────────────────────────────────────────────────
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
    print(f"Usando ruta configurada: {DATA_ROOT}")
else:
    _cwd = Path(os.getcwd())
    _candidates = [
        _cwd / ".." / "data",
        _cwd / "..",
        _cwd / "data",
        _cwd,
        Path("/content/drive/MyDrive/landslide4sense"),
        Path("/content/drive/MyDrive/Landslide_ML"),
        Path("/content/landslide4sense"),
        Path("/content/data"),
        Path("/content"),
    ]
    DATA_ROOT = None
    for _c in _candidates:
        if (_c / "TrainData" / "img").exists():
            DATA_ROOT = _c.resolve()
            print(f"Dataset detectado automáticamente en: {DATA_ROOT}")
            break

if DATA_ROOT is None or not (DATA_ROOT / "TrainData" / "img").exists():
    print("""
⚠️  Dataset no encontrado. Opciones:

━━━  OPCIÓN A — Google Drive (recomendado)  ━━━
  Ejecuta la celda de Drive de arriba y vuelve a correr esta celda.

━━━  OPCIÓN B — Kaggle API  ━━━
  !pip install kaggle -q
  # Sube tu kaggle.json y ejecuta:
  !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
  !kaggle datasets download -d landslide4sense/competition -p /content/
  !unzip -q /content/competition.zip -d /content/landslide4sense/
  DATA_ROOT_OVERRIDE = "/content/landslide4sense"
""")
    raise FileNotFoundError("Dataset no encontrado. Establece DATA_ROOT_OVERRIDE con la ruta correcta.")

# ── Verificar estructura del dataset ─────────────────────────────────────────
n_train = len(list((DATA_ROOT / "TrainData" / "img").glob("*.h5")))
print(f"✓ Dataset encontrado en: {DATA_ROOT}")
print(f"  TrainData/img : {n_train} archivos .h5")


## 1. Balance de Clases

In [ ]:
labels, areas = [], []
for f in tqdm(img_files, desc="Cargando etiquetas"):
    mf = mask_dir / f.name
    with h5py.File(mf,"r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    lbl = int(mask.max() > 0)
    area = float(mask.mean())
    labels.append(lbl); areas.append(area)

labels = np.array(labels); areas = np.array(areas)
n_pos = labels.sum(); n_neg = (1-labels).sum()
print(f"Total parches: {len(labels)}")
print(f"Positivos (deslizamiento):     {n_pos} ({100*n_pos/len(labels):.1f}%)")
print(f"Negativos (no-deslizamiento): {n_neg} ({100*n_neg/len(labels):.1f}%)")
print(f"Ratio n_neg/n_pos:            {n_neg/n_pos:.3f}  ← pos_weight para BCE")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Pie chart
axes[0].pie([n_pos, n_neg], labels=["Deslizamiento\n(58.7%)", "No-desliz.\n(41.3%)"],
            colors=["#e74c3c","#3498db"], autopct="%1.1f%%", startangle=90,
            textprops={"fontsize":11})
axes[0].set_title("Balance de Clases", fontsize=13, fontweight="bold")

# Histograma de áreas (solo positivos)
pos_areas = areas[labels==1] * 100  # Porcentaje
axes[1].hist(pos_areas, bins=40, color="#e74c3c", alpha=0.85, edgecolor="white")
axes[1].axvline(np.median(pos_areas), color="black", ls="--", lw=2,
                label=f"Mediana: {np.median(pos_areas):.2f}%")
axes[1].axvline(np.percentile(pos_areas,75), color="orange", ls="--", lw=1.5,
                label=f"P75: {np.percentile(pos_areas,75):.2f}%")
axes[1].set_xlabel("Área del deslizamiento (% del parche)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de Áreas (parches positivos)", fontsize=12, fontweight="bold")
axes[1].legend()

# CDF
sorted_areas = np.sort(pos_areas)
cdf = np.arange(1, len(sorted_areas)+1) / len(sorted_areas)
axes[2].plot(sorted_areas, cdf, color="#e74c3c", lw=2)
axes[2].axvline(2.04, color="black", ls="--", lw=1.5, label="Mediana (2.04%)")
axes[2].set_xlabel("Área del deslizamiento (%)")
axes[2].set_ylabel("CDF")
axes[2].set_title("Función de Distribución Acumulada (CDF)", fontsize=12, fontweight="bold")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle("EDA — Balance y Distribución de Áreas", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../docs/figures/fig2_class_balance_areas.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Área mediana: {np.median(pos_areas):.2f}%  ← Small Object Detection challenge")

## 2. Discriminabilidad por Canal

In [ ]:
# Calcular media por clase para cada canal sobre muestra
step = max(1, len(img_files) // N_SAMPLE)
sampled = img_files[::step][:N_SAMPLE]

mean_pos = np.zeros(N_CHANNELS)
mean_neg = np.zeros(N_CHANNELS)
cnt_pos = cnt_neg = 0

for f in tqdm(sampled, desc="Estadísticas por canal"):
    mf = mask_dir / f.name
    with h5py.File(f,  "r") as hf: patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mf, "r") as hf: mask  = hf[list(hf.keys())[0]][()]
    lbl = int(mask.max() > 0)
    ch_means = patch.mean(axis=(0,1))
    if lbl == 1: mean_pos += ch_means; cnt_pos += 1
    else:         mean_neg += ch_means; cnt_neg += 1

mean_pos /= cnt_pos; mean_neg /= cnt_neg
delta = mean_pos - mean_neg
print("Top-5 canales más discriminativos:")
top5 = np.argsort(np.abs(delta))[-5:][::-1]
for rank, ch in enumerate(top5):
    print(f"  {rank+1}. Ch{ch:02d} {CHANNEL_NAMES[ch]:20s}  Δ={delta[ch]:+.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 9))
x = np.arange(N_CHANNELS); w = 0.35
names = [f"Ch{i:02d}\n{n.split(' ')[0]}" for i,n in enumerate(CHANNEL_NAMES)]

axes[0].bar(x-w/2, mean_pos, w, label="Deslizamiento", color="#e74c3c", alpha=0.85)
axes[0].bar(x+w/2, mean_neg, w, label="No-deslizamiento", color="#3498db", alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=35, ha="right", fontsize=9)
axes[0].set_title("Media por Canal por Clase", fontweight="bold"); axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

colors = ["#e74c3c" if d>0 else "#3498db" for d in delta]
axes[1].bar(x, delta, color=colors, alpha=0.85)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=35, ha="right", fontsize=9)
axes[1].set_ylabel("Δ Media (Pos − Neg)"); axes[1].set_title("Diferencia Discriminativa por Canal", fontweight="bold")
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("EDA — Discriminabilidad Espectral por Canal", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../docs/figures/fig3_channel_class_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Correlación entre Canales

In [ ]:
# Matriz de correlación de Pearson
step2 = max(1, len(img_files) // 30)
channel_data = []
for f in img_files[::step2][:30]:
    with h5py.File(f,"r") as hf:
        patch = hf[list(hf.keys())[0]][()]
    # Muestrear píxeles
    flat = patch.reshape(-1, N_CHANNELS)[::8]
    channel_data.append(flat)

data_arr = np.vstack(channel_data)
corr_matrix = np.corrcoef(data_arr.T)

fig, ax = plt.subplots(figsize=(12, 10))
short_names = [f"Ch{i:02d}" for i in range(N_CHANNELS)]
mask_tri = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=ax, xticklabels=short_names, yticklabels=short_names,
            linewidths=0.3, square=True, cbar_kws={"label":"Pearson r"})
ax.set_title("Correlación de Pearson entre Canales (14×14)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../docs/figures/fig5_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Muestras Positivas y Negativas

In [ ]:
def clip_pct(arr, p=2):
    lo, hi = np.percentile(arr, p), np.percentile(arr, 100-p)
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)

pos_files = [f for f,l in zip(img_files, labels) if l==1][:4]
neg_files = [f for f,l in zip(img_files, labels) if l==0][:4]

fig, axes = plt.subplots(4, 6, figsize=(20, 14))
for row, (flist, cls_name) in enumerate([(pos_files,"Positivo"),(neg_files,"Negativo")]):
    for col, f in enumerate(flist):
        mf = mask_dir / f.name
        with h5py.File(f,"r") as hf:  patch = hf[list(hf.keys())[0]][()]
        with h5py.File(mf,"r") as hf: mask_  = hf[list(hf.keys())[0]][()]
        rgb = clip_pct(patch[:,:,[2,1,0]])
        nir = clip_pct(patch[:,:,[3,2,1]])
        sar = clip_pct(patch[:,:,7])
        axes[row*2][col].imshow(rgb); axes[row*2][col].axis("off")
        axes[row*2][col].set_title(f"{cls_name}\nRGB", fontsize=9)
        axes[row*2+1][col].imshow(sar, cmap="gray"); axes[row*2+1][col].axis("off")
        axes[row*2+1][col].set_title("SAR VV", fontsize=9)

plt.suptitle("EDA — Muestras Positivas (Deslizamiento) y Negativas", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../docs/figures/fig1_samples_pos_neg.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Resumen EDA — Implicaciones para el Diseño Experimental

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║  RESUMEN EDA — Hallazgos Clave                                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. BALANCE DE CLASES (casi equiparado)                          ║
║     Positivos: 58.7% | Negativos: 41.3%                         ║
║     → pos_weight = 0.703 en BCEWithLogitsLoss                   ║
║     → No se requiere over/undersampling                         ║
║                                                                  ║
║  2. DESLIZAMIENTOS DE PEQUEÑA ESCALA                             ║
║     Área mediana: 2.04% del parche (~334 px / 16,384)           ║
║     Percentil 90: 9.89%                                          ║
║     → U-Net crítico para localización precisa                   ║
║     → Dice Loss necesario para segmentación (objetos pequeños)  ║
║                                                                  ║
║  3. CANALES MÁS DISCRIMINATIVOS                                  ║
║     Ch13 RedEdge3: Δ=+0.807 (estrés vegetal)                    ║
║     Ch12 RedEdge2: Δ=+0.563 (cambio en dosel)                   ║
║     Ch09 DEM:      Δ=+0.195 (mayor elevación en eventos)        ║
║     → Fusión multiespectral es fundamental                      ║
║     → Canales Red-Edge son los más informativos                 ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")